# House Prices — improved simple polynomial regression
This notebook uses richer numeric and domain features, a log-transformed target, documented outlier handling, degree comparison, and residual analysis. The estimator remains `LinearRegression` on polynomial features.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.modeling import (
    MODEL_FEATURES, TARGET, cross_validate_degrees, fit_model,
    outlier_mask, predict_prices, prepare_features, regression_metrics,
)

## Load and inspect the data

In [ ]:
train = pd.read_csv(ROOT / 'train.csv')
test = pd.read_csv(ROOT / 'test.csv')
print('Train:', train.shape, 'Test:', test.shape)
train['SalePrice'].describe()

## Documented outlier handling and feature engineering
The rule flags houses above 4,000 sq ft that sold below $300,000. We report and remove these unusual training observations rather than hiding the decision.

In [ ]:
flagged = outlier_mask(train)
display(train.loc[flagged, ['Id', 'GrLivArea', 'SalePrice']])
modeling_train = train.loc[~flagged].reset_index(drop=True)
X = prepare_features(modeling_train)
y = modeling_train[TARGET]
print('Rows retained:', len(modeling_train))
print('Model features:', MODEL_FEATURES)
X.head()

## Five-fold comparison: degree 1 versus degree 2
This is a transparent comparison within the polynomial-regression family. Degree 2 remains the final educational model so squared and interaction terms stay central to the project.

In [ ]:
degree_results = pd.DataFrame(cross_validate_degrees(X, y, degrees=(1, 2), n_splits=5))
degree_results

## Holdout evaluation of degree 2
Preprocessing is learned from the training split only. The target uses `log1p`, and predictions are returned to dollars with `expm1`.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)
fitted = fit_model(X_train, y_train, degree=2)
valid_pred = predict_prices(fitted, X_valid)
regression_metrics(y_valid, valid_pred)

## Residual diagnostics

In [ ]:
diagnostics = pd.DataFrame({'Actual': y_valid, 'Predicted': valid_pred})
diagnostics['Residual'] = diagnostics['Actual'] - diagnostics['Predicted']
diagnostics['AbsoluteError'] = diagnostics['Residual'].abs()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(diagnostics['Predicted'], diagnostics['Residual'], alpha=0.6)
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set(xlabel='Predicted price', ylabel='Residual', title='Residuals vs predicted')
axes[1].scatter(diagnostics['Actual'], diagnostics['Predicted'], alpha=0.6)
low = min(diagnostics['Actual'].min(), diagnostics['Predicted'].min())
high = max(diagnostics['Actual'].max(), diagnostics['Predicted'].max())
axes[1].plot([low, high], [low, high], '--', color='black')
axes[1].set(xlabel='Actual price', ylabel='Predicted price', title='Actual vs predicted')
plt.tight_layout()
plt.show()
diagnostics.nlargest(10, 'AbsoluteError')

## Refit degree 2 and create the Kaggle submission

In [ ]:
final_fitted = fit_model(X, y, degree=2)
test_pred = predict_prices(final_fitted, prepare_features(test))
submission = pd.DataFrame({'Id': test['Id'], TARGET: test_pred})
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
submission.to_csv(OUTPUTS / 'polynomial_submission.csv', index=False)
submission.head()

## Conclusion
The project still uses only polynomial regression. The stronger data treatment improves the holdout result, while cross-validation honestly shows whether degree 2 adds value over the degree-1 baseline.